# 75 — Grand Metrics Comparison

Loads OOF predictions from ALL available models and computes RAE, MAE, R², Pearson r, Spearman ρ, Kendall τ, and cliff-pair accuracy for every model. Ranks them and identifies which data combinations and model types move the needle on activity cliffs specifically.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED,
    verbose=-1, n_jobs=4,
)


In [2]:
def full_metrics(y_true, y_pred, cliff_pairs_df=None, label=""):
    """RAE, MAE, R², Pearson, Spearman, Kendall, Cliff_accuracy."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    mae_v  = float(np.mean(np.abs(yt - yp)))
    rae_v  = mae_v / float(np.mean(np.abs(yt - yt.mean()))) if yt.std() > 0 else float("nan")
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2_v   = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    pr_v, _ = stats.pearsonr(yt, yp)
    sp_v, _ = stats.spearmanr(yt, yp)
    kt_v, _ = stats.kendalltau(yt, yp)

    m = dict(RAE=rae_v, MAE=mae_v, R2=r2_v,
             Pearson=pr_v, Spearman=sp_v, Kendall=kt_v)

    if cliff_pairs_df is not None and len(cliff_pairs_df) > 0:
        correct = total = 0
        for _, row in cliff_pairs_df.iterrows():
            ia, ii = int(row.get("idx_active", -1)), int(row.get("idx_inactive", -1))
            if 0 <= ia < len(yp) and 0 <= ii < len(yp):
                correct += int(yp[ia] > yp[ii])
                total   += 1
        m["Cliff_acc"] = correct / total if total else float("nan")

    if label:
        cliff_str = f"  Cliff_acc={m.get('Cliff_acc', float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f}  MAE={mae_v:.4f}  R²={r2_v:.4f}  "
              f"Pearson={pr_v:.4f}  Spearman={sp_v:.4f}  Kendall={kt_v:.4f}{cliff_str}")
    return m


In [3]:
tr = load_train()
te = load_test()
print(f"CRC train: {len(tr):,}  |  Test: {len(te):,}")

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
active_mask = y_tr >= 5.5
print(f"X_tr: {X_tr.shape}  actives: {active_mask.sum()}")

cliff_pairs = (pd.read_parquet(DATA_PROCESSED / "cliff_pairs.parquet")
               if (DATA_PROCESSED / "cliff_pairs.parquet").exists()
               else pd.DataFrame())
print(f"Cliff pairs available: {len(cliff_pairs)}")


CRC train: 4,139  |  Test: 513


X_tr: (4139, 2265)  actives: 380
Cliff pairs available: 149


In [4]:
# Discover all OOF files
oof_files = sorted(DATA_PROCESSED.glob("oof_*.npy"))
print(f"Found {len(oof_files)} OOF files")

# Add idx_active/idx_inactive to cliff_pairs if available
if len(cliff_pairs) > 0:
    smiles_to_idx = {smi: i for i, smi in enumerate(tr["smiles"].tolist())}
    act_col = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ina_col = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    if act_col in cliff_pairs.columns and "idx_active" not in cliff_pairs.columns:
        cliff_pairs["idx_active"]   = cliff_pairs[act_col].map(smiles_to_idx)
        cliff_pairs["idx_inactive"] = cliff_pairs[ina_col].map(smiles_to_idx)
        cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
        cliff_pairs["idx_active"]   = cliff_pairs["idx_active"].astype(int)
        cliff_pairs["idx_inactive"] = cliff_pairs["idx_inactive"].astype(int)
    print(f"Cliff pairs with indices: {len(cliff_pairs)}")

rows = []
for fp in oof_files:
    name = fp.stem.replace("oof_", "")
    try:
        oof = np.load(fp)
        # Handle multi-column OOF (multitask models) — take primary column
        if oof.ndim > 1:
            oof = oof[:, 0]
        if len(oof) != len(y_tr):
            print(f"  SKIP {name}: shape mismatch ({len(oof)} vs {len(y_tr)})")
            continue
        if not np.isfinite(oof).all():
            n_nan = np.isnan(oof).sum()
            print(f"  {name}: {n_nan} NaN values — filling with mean")
            oof[~np.isfinite(oof)] = y_tr.mean()
        m = full_metrics(y_tr, oof, cliff_pairs if len(cliff_pairs) > 0 else None)
        m["model"] = name
        m["n_valid"] = int(np.isfinite(oof).sum())
        rows.append(m)
    except Exception as e:
        print(f"  ERROR {name}: {e}")

df = pd.DataFrame(rows).set_index("model")
df = df.sort_values("RAE")
print(f"\n{'='*80}")
print("ALL MODELS — sorted by RAE (lower is better)")
print(f"{'='*80}")
print(df[["RAE","MAE","R2","Pearson","Spearman","Kendall"]
         + (["Cliff_acc"] if "Cliff_acc" in df.columns else [])].round(4).to_string())

Found 61 OOF files
Cliff pairs with indices: 0
  SKIP chemprop_aux_BAD4141: shape mismatch (4141 vs 4139)



ALL MODELS — sorted by RAE (lower is better)
                                 RAE     MAE       R2  Pearson  Spearman  Kendall
model                                                                            
aux_features                  0.2179  0.1982   0.9348   0.9669    0.9472   0.8201
grand_v6                      0.2189  0.1991   0.9347   0.9668    0.9473   0.8198
grand_v7                      0.5189  0.4721   0.6565   0.8103    0.7666   0.5733
grand_v6c                     0.5281  0.4804   0.6481   0.8051    0.7620   0.5679
grand_v6b                     0.5281  0.4804   0.6481   0.8051    0.7620   0.5679
grand25                       0.5356  0.4873   0.6400   0.8000    0.7547   0.5608
grand24                       0.5358  0.4875   0.6399   0.8000    0.7546   0.5606
grand23                       0.5360  0.4877   0.6398   0.7999    0.7544   0.5603
grand18                       0.5363  0.4879   0.6392   0.7995    0.7538   0.5600
lgbm_tuned                    0.5394  0.4908   0.634

In [5]:
# Highlight cliff accuracy specifically
if "Cliff_acc" in df.columns:
    cliff_rank = df["Cliff_acc"].dropna().sort_values(ascending=False)
    if len(cliff_rank) > 0:
        print("\n=== Cliff Accuracy Ranking (higher is better) ===")
        print(cliff_rank.round(3).to_string())
        print(f"\nRandom baseline cliff accuracy: ~0.500")
        print(f"Best model cliff accuracy: {cliff_rank.iloc[0]:.3f} ({cliff_rank.index[0]})")
        print(f"Worst model cliff accuracy: {cliff_rank.iloc[-1]:.3f} ({cliff_rank.index[-1]})")
    else:
        print("\nCliff_acc column present but all NaN — cliff pair indices not resolved yet. Run nb61 first.")

# Active compound subset metrics
print("\n=== Active Subset (pEC50 ≥ 5.5) Ranking ===")
rows_active = []
for fp in sorted(DATA_PROCESSED.glob("oof_*.npy")):
    name = fp.stem.replace("oof_", "")
    try:
        oof = np.load(fp)
        if oof.ndim > 1:
            oof = oof[:, 0]
        if len(oof) != len(y_tr): continue
        oof[~np.isfinite(oof)] = y_tr.mean()
        m = full_metrics(y_tr[active_mask], oof[active_mask])
        m["model"] = name
        rows_active.append(m)
    except: pass
df_active = pd.DataFrame(rows_active).set_index("model").sort_values("RAE")
print(df_active[["RAE","MAE","Pearson","Spearman"]].round(4).head(20).to_string())


=== Active Subset (pEC50 ≥ 5.5) Ranking ===


                                RAE     MAE  Pearson  Spearman
model                                                         
grand_v6                     1.5775  0.3308   0.3483    0.3536
aux_features                 1.5873  0.3329   0.3551    0.3570
lgbm_all_external_v2         2.7018  0.5666   0.1842    0.1863
lgbm_crc_sp_chembl_pxr       2.7558  0.5779   0.1535    0.0413
nr_weighted                  3.2221  0.6757   0.2064    0.1796
catboost                     3.2298  0.6773   0.0197    0.0704
lgbm_cliff_aware_external    3.3075  0.6936   0.1542    0.1456
lgbm_chembl_all_nr_weighted  3.3144  0.6950   0.1627    0.1581
wide_deep                    3.3878  0.7104  -0.0044    0.0310
morgan_protbert_nr           3.4224  0.7177   0.1944    0.1271
chemprop_cliff               3.4284  0.7189  -0.0409    0.0617
lgbm_chembl_pxr_direct       3.4316  0.7196   0.1805    0.1302
morgan_esm2_nr               3.4319  0.7197   0.2218    0.1480
siamese_cliff                3.4450  0.7224  -0.0289   

In [6]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metrics_to_plot = ["RAE","MAE","R2","Pearson","Spearman","Kendall"]
colors = ["red" if v == df[m].min() else "steelblue"
          for m, v in zip(["RAE","MAE"] + ["R2","Pearson","Spearman","Kendall"] * 2,
                          [df["RAE"].min(), df["MAE"].min()] + [0]*4)]

for ax, metric in zip(axes.flat, metrics_to_plot):
    ascending = metric in ("RAE","MAE")
    top = df[metric].dropna().sort_values(ascending=ascending).head(20)
    colors_m = ["#d62728" if i == 0 else "#1f77b4" for i in range(len(top))]
    ax.barh(top.index[::-1], top.values[::-1], color=colors_m[::-1])
    ax.set_title(metric); ax.set_xlabel(metric)
    ax.axvline(0, color="black", linewidth=0.5)

plt.suptitle("Model Comparison — All Metrics (top 20 each)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig_path = DATA_PROCESSED / "figures" / "75_grand_metrics_comparison.png"
fig_path.parent.mkdir(exist_ok=True)
plt.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.close()
print(f"Saved figure: {fig_path}")


Saved figure: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\figures\75_grand_metrics_comparison.png


In [7]:
# Save results table
out_csv = DATA_PROCESSED / "all_model_metrics.csv"
df.reset_index().to_csv(out_csv, index=False)
print(f"Saved metrics table: {out_csv}")
print(f"\nTop-5 by RAE:")
print(df.head(5)[["RAE","MAE","R2","Pearson","Spearman","Kendall"]].round(4).to_string())


Saved metrics table: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\all_model_metrics.csv

Top-5 by RAE:
                 RAE     MAE      R2  Pearson  Spearman  Kendall
model                                                           
aux_features  0.2179  0.1982  0.9348   0.9669    0.9472   0.8201
grand_v6      0.2189  0.1991  0.9347   0.9668    0.9473   0.8198
grand_v7      0.5189  0.4721  0.6565   0.8103    0.7666   0.5733
grand_v6c     0.5281  0.4804  0.6481   0.8051    0.7620   0.5679
grand_v6b     0.5281  0.4804  0.6481   0.8051    0.7620   0.5679
